In [1]:
!pip -q install scikit-learn xgboost scipy

In [2]:
import os
import numpy as np
import pandas as pd

from scipy.spatial.distance import cosine

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from xgboost import XGBRegressor

In [3]:
os.makedirs("data/11_round2_text_change", exist_ok=True)

In [4]:
round1_df = pd.read_csv("round1_improved_baseline_dataset.csv")
aligned_df = pd.read_csv("dataset_v2_aligned_finbert_updated.csv")
raw_emb = np.load("finbert_embeddings_raw_updated.npy")

print("Round 1 shape:", round1_df.shape)
print("Aligned Step 7 shape:", aligned_df.shape)
print("Raw embeddings shape:", raw_emb.shape)

print("aligned_df rows:", len(aligned_df))
print("raw_emb rows:", raw_emb.shape[0])
assert len(aligned_df) == raw_emb.shape[0], "Embedding rows do not match aligned_df rows"

Round 1 shape: (2181, 41)
Aligned Step 7 shape: (2247, 26)
Raw embeddings shape: (2247, 768)
aligned_df rows: 2247
raw_emb rows: 2247


In [5]:
for df in [round1_df, aligned_df]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce")

if "accession_number" in round1_df.columns:
    round1_df["accession_number"] = round1_df["accession_number"].astype(str).str.strip()

if "accession_number" in aligned_df.columns:
    aligned_df["accession_number"] = aligned_df["accession_number"].astype(str).str.strip()

In [6]:
aligned_df = aligned_df.copy()
aligned_df["emb_idx"] = np.arange(len(aligned_df))

merge_keys = ["ticker", "filing_date", "filing_type"]
if "accession_number" in round1_df.columns and "accession_number" in aligned_df.columns:
    merge_keys.append("accession_number")

round2_df = round1_df.merge(
    aligned_df[merge_keys + ["emb_idx"]],
    on=merge_keys,
    how="left"
)

print("Round 2 merged shape:", round2_df.shape)
print("Missing emb_idx rows:", round2_df["emb_idx"].isna().sum())
round2_df.head()

Round 2 merged shape: (2181, 42)
Missing emb_idx rows: 0


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,mean_abs_return_5d,mean_abs_return_10d,max_abs_return_10d,return_skew_10d,return_kurtosis_10d,log_price_tminus1,vol_ratio_5_20,vol_ratio_10_60,log_future_realized_vol_10d,emb_idx
0,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2,320193,32019319000066,AAPL_20190501_10-Q_000032019319000066.txt,...,0.007233,0.007704,0.019473,-0.006040,0.587393,3.915367,0.870016,0.974208,-3.810629,1
1,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,AAPL_20190731_10-Q_000032019319000076.txt,...,0.005166,0.008841,0.022854,0.360768,-0.039101,3.954987,0.662464,0.709436,-3.566474,2
2,AAPL,320193,2019-10-31,10-K,0000320193-19-000119,2019,4,320193,32019319000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...,...,0.009446,0.008896,0.023128,-1.456441,3.091214,4.107836,1.159828,0.733736,-4.628291,2245
3,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,AAPL_20200129_10-Q_000032019320000010.txt,...,0.013792,0.011713,0.029405,-0.158909,0.862831,4.374782,1.468833,1.343708,-3.852132,3
4,AAPL,320193,2020-05-01,10-Q,0000320193-20-000052,2020,2,320193,32019320000052,AAPL_20200501_10-Q_000032019320000052.txt,...,0.019945,0.019764,0.032845,0.076090,-1.717656,4.296605,0.738856,0.523305,-4.397591,4


In [7]:
round2_df = round2_df.dropna(subset=["emb_idx"]).copy()
round2_df["emb_idx"] = round2_df["emb_idx"].astype(int)

print("Round 2 shape after keeping embedding rows:", round2_df.shape)

Round 2 shape after keeping embedding rows: (2181, 42)


In [8]:
round2_df = round2_df.sort_values(["ticker", "filing_date"]).reset_index(drop=True)
round2_df.head()

,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,mean_abs_return_5d,mean_abs_return_10d,max_abs_return_10d,return_skew_10d,return_kurtosis_10d,log_price_tminus1,vol_ratio_5_20,vol_ratio_10_60,log_future_realized_vol_10d,emb_idx
0,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2,320193,32019319000066,AAPL_20190501_10-Q_000032019319000066.txt,...,0.007233,0.007704,0.019473,-0.006040,0.587393,3.915367,0.870016,0.974208,-3.810629,1
1,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,AAPL_20190731_10-Q_000032019319000076.txt,...,0.005166,0.008841,0.022854,0.360768,-0.039101,3.954987,0.662464,0.709436,-3.566474,2
2,AAPL,320193,2019-10-31,10-K,0000320193-19-000119,2019,4,320193,32019319000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...,...,0.009446,0.008896,0.023128,-1.456441,3.091214,4.107836,1.159828,0.733736,-4.628291,2245
3,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,AAPL_20200129_10-Q_000032019320000010.txt,...,0.013792,0.011713,0.029405,-0.158909,0.862831,4.374782,1.468833,1.343708,-3.852132,3
4,AAPL,320193,2020-05-01,10-Q,0000320193-20-000052,2020,2,320193,32019320000052,AAPL_20200501_10-Q_000032019320000052.txt,...,0.019945,0.019764,0.032845,0.076090,-1.717656,4.296605,0.738856,0.523305,-4.397591,4


In [9]:
change_cols = [
    "lm_negative",
    "lm_positive",
    "lm_uncertainty",
    "lm_net_sentiment",
    "log_text_length_words"
]

for col in change_cols:
    round2_df[f"prev_{col}"] = round2_df.groupby("ticker")[col].shift(1)
    round2_df[f"delta_{col}"] = round2_df[col] - round2_df[f"prev_{col}"]

round2_df[[
    "ticker", "filing_date",
    "delta_lm_negative",
    "delta_lm_positive",
    "delta_lm_uncertainty",
    "delta_lm_net_sentiment",
    "delta_log_text_length_words"
]].head(10)

,ticker,filing_date,delta_lm_negative,delta_lm_positive,delta_lm_uncertainty,delta_lm_net_sentiment,delta_log_text_length_words
0,AAPL,2019-05-01,NaN,NaN,NaN,NaN,NaN
1,AAPL,2019-07-31,0.000531,0.000070,0.000689,-0.000461,-0.014588
2,AAPL,2019-10-31,-0.030837,-0.003492,-0.020055,0.027345,-2.071392
3,AAPL,2020-01-29,-0.004313,-0.000213,-0.004458,0.004100,1.339034
4,AAPL,2020-05-01,0.003296,0.001141,-0.000785,-0.002154,0.270599
5,AAPL,2020-07-31,-0.001517,-0.001525,0.000449,-0.000008,-0.011531
6,AAPL,2020-10-30,0.005114,0.000033,0.004055,-0.005081,-1.602972
7,AAPL,2021-01-28,-0.009599,0.001330,-0.002388,0.010928,1.385806
8,AAPL,2021-04-29,0.000460,0.000970,-0.001050,0.000510,0.121521
9,AAPL,2021-07-28,-0.000407,0.001193,-0.000311,0.001600,0.044217


In [10]:
round2_df["prev_emb_idx"] = round2_df.groupby("ticker")["emb_idx"].shift(1)

print("Rows without previous filing:", round2_df["prev_emb_idx"].isna().sum())

Rows without previous filing: 123


In [11]:
def cosine_similarity_safe(a, b):
    if a is None or b is None:
        return np.nan
    if np.isnan(a).any() or np.isnan(b).any():
        return np.nan
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return np.nan
    return 1 - cosine(a, b)

def l2_distance_safe(a, b):
    if a is None or b is None:
        return np.nan
    if np.isnan(a).any() or np.isnan(b).any():
        return np.nan
    return np.linalg.norm(a - b)

In [12]:
cos_sims = []
l2_dists = []

for _, row in round2_df.iterrows():
    if pd.isna(row["prev_emb_idx"]):
        cos_sims.append(np.nan)
        l2_dists.append(np.nan)
        continue

    curr_vec = raw_emb[int(row["emb_idx"])]
    prev_vec = raw_emb[int(row["prev_emb_idx"])]

    cos_sims.append(cosine_similarity_safe(curr_vec, prev_vec))
    l2_dists.append(l2_distance_safe(curr_vec, prev_vec))

round2_df["finbert_cosine_prev"] = cos_sims
round2_df["finbert_l2_prev"] = l2_dists

round2_df[["finbert_cosine_prev", "finbert_l2_prev"]].describe()

,finbert_cosine_prev,finbert_l2_prev
count,2058.000000,2058.000000
mean,0.985118,1.294443
std,0.026296,1.084039
min,0.814844,0.000000
25%,0.986022,0.576144
50%,0.994961,0.956764
75%,0.998183,1.567260
max,1.000000,6.100934


In [13]:
round2_df = round2_df.dropna(subset=[
    "delta_lm_negative",
    "delta_lm_positive",
    "delta_lm_uncertainty",
    "delta_lm_net_sentiment",
    "delta_log_text_length_words",
    "finbert_cosine_prev",
    "finbert_l2_prev"
]).copy()

print("Round 2 final dataset shape:", round2_df.shape)
round2_df.head()

Round 2 final dataset shape: (2058, 55)


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,delta_lm_positive,prev_lm_uncertainty,delta_lm_uncertainty,prev_lm_net_sentiment,delta_lm_net_sentiment,prev_log_text_length_words,delta_log_text_length_words,prev_emb_idx,finbert_cosine_prev,finbert_l2_prev
1,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,AAPL_20190731_10-Q_000032019319000076.txt,...,0.000070,0.033989,0.000689,-0.034936,-0.000461,9.022323,-0.014588,1.0,0.999999,0.014831
2,AAPL,320193,2019-10-31,10-K,0000320193-19-000119,2019,4,320193,32019319000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...,...,-0.003492,0.034677,-0.020055,-0.035397,0.027345,9.007734,-2.071392,2.0,0.883218,4.649836
3,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,AAPL_20200129_10-Q_000032019320000010.txt,...,-0.000213,0.014622,-0.004458,-0.008053,0.004100,6.936343,1.339034,2245.0,0.994290,1.021676
4,AAPL,320193,2020-05-01,10-Q,0000320193-20-000052,2020,2,320193,32019320000052,AAPL_20200501_10-Q_000032019320000052.txt,...,0.001141,0.010164,-0.000785,-0.003953,-0.002154,8.275376,0.270599,3.0,0.977487,2.046331
5,AAPL,320193,2020-07-31,10-Q,0000320193-20-000062,2020,3,320193,32019320000062,AAPL_20200731_10-Q_000032019320000062.txt,...,-0.001525,0.009378,0.000449,-0.006107,-0.000008,8.545975,-0.011531,4.0,0.993636,1.063818


In [14]:
print("Round 1 rows:", len(round1_df))
print("Round 2 rows:", len(round2_df))
print("Rows lost:", len(round1_df) - len(round2_df))

Round 1 rows: 2181
Round 2 rows: 2058
Rows lost: 123


In [15]:
round2_df.to_csv("data/11_round2_text_change/round2_text_change_dataset.csv", index=False)
print("Saved Round 2 dataset.")

Saved Round 2 dataset.


In [16]:
split_date = pd.Timestamp("2023-01-01")

train_df = round2_df.loc[round2_df["filing_date"] < split_date].copy().reset_index(drop=True)
test_df = round2_df.loc[round2_df["filing_date"] >= split_date].copy().reset_index(drop=True)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train range:", train_df["filing_date"].min(), "to", train_df["filing_date"].max())
print("Test range:", test_df["filing_date"].min(), "to", test_df["filing_date"].max())

Train rows: 1272
Test rows: 786
Train range: 2019-05-31 00:00:00 to 2022-12-29 00:00:00
Test range: 2023-01-05 00:00:00 to 2024-12-13 00:00:00


In [17]:
price_plus_context_features = [
    "past_return_5d",
    "past_return_10d",
    "past_return_20d",
    "abs_past_return_10d",
    "past_realized_vol_5d",
    "past_realized_vol_10d",
    "past_realized_vol_20d",
    "past_realized_vol_30d",
    "past_realized_vol_60d",
    "mean_abs_return_5d",
    "mean_abs_return_10d",
    "max_abs_return_10d",
    "return_skew_10d",
    "return_kurtosis_10d",
    "log_price_tminus1",
    "vol_ratio_5_20",
    "vol_ratio_10_60",
    "is_10k",
    "log_text_length_words"
]

categorical_features = ["quarter", "filing_month", "filing_year"]

lm_level_features = [
    "lm_negative",
    "lm_positive",
    "lm_uncertainty",
    "lm_net_sentiment"
]

lm_change_features = [
    "delta_lm_negative",
    "delta_lm_positive",
    "delta_lm_uncertainty",
    "delta_lm_net_sentiment",
    "delta_log_text_length_words"
]

finbert_change_features = [
    "finbert_cosine_prev",
    "finbert_l2_prev"
]

target_col = "log_future_realized_vol_10d"

feature_sets = {
    "round2_baseline_price_context": price_plus_context_features + categorical_features,
    "round2_plus_lm_levels": price_plus_context_features + lm_level_features + categorical_features,
    "round2_plus_text_change": price_plus_context_features + lm_level_features + lm_change_features + finbert_change_features + categorical_features
}

In [18]:
def evaluate_regression_both_scales(y_true_log, y_pred_log):
    # log scale metrics
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    r2_log = r2_score(y_true_log, y_pred_log)

    # original scale metrics
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "RMSE_log": rmse_log,
        "MAE_log": mae_log,
        "R2_log": r2_log,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }

In [19]:
y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

naive_pred = np.repeat(y_train.mean(), len(y_test))
naive_metrics = evaluate_regression_both_scales(y_test, naive_pred)

naive_results_df = pd.DataFrame([{
    "model": "NaiveMean",
    "dataset": "naive_mean",
    **naive_metrics
}])

naive_results_df

,model,dataset,RMSE_log,MAE_log,R2_log,RMSE,MAE,R2
0,NaiveMean,naive_mean,0.555191,0.451961,-0.291755,0.010143,0.006944,-0.046412


In [20]:
all_results = []

for dataset_name, cols in feature_sets.items():
    X_train = train_df[cols].copy()
    X_test = test_df[cols].copy()

    y_train = train_df[target_col].copy()
    y_test = test_df[target_col].copy()

    numeric_cols = [c for c in cols if c not in categorical_features]
    categorical_cols = categorical_features.copy()

    linear_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical_cols)
        ]
    )

    tree_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical_cols)
        ]
    )

    models = {
        "LinearRegression": Pipeline([
            ("prep", linear_preprocessor),
            ("model", LinearRegression())
        ]),
        "ElasticNet": Pipeline([
            ("prep", linear_preprocessor),
            ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42))
        ]),
        "RandomForest": Pipeline([
            ("prep", tree_preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=400,
                max_depth=10,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "XGBoost": Pipeline([
            ("prep", tree_preprocessor),
            ("model", XGBRegressor(
                n_estimators=400,
                max_depth=4,
                learning_rate=0.03,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_alpha=0.0,
                reg_lambda=1.0,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            ))
        ])
    }

    for model_name, pipe in models.items():
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        metrics = evaluate_regression_both_scales(y_test, preds)
        metrics["model"] = model_name
        metrics["dataset"] = dataset_name
        all_results.append(metrics)

results_df = pd.DataFrame(all_results)
results_df = pd.concat([naive_results_df, results_df], ignore_index=True)
results_df = results_df.sort_values("RMSE").reset_index(drop=True)

results_df

,model,dataset,RMSE_log,MAE_log,R2_log,RMSE,MAE,R2
0,RandomForest,round2_baseline_price_context,0.436865,0.347326,0.200187,0.008795,0.005404,0.213317
1,RandomForest,round2_plus_text_change,0.438460,0.347871,0.194336,0.008813,0.005396,0.210033
2,RandomForest,round2_plus_lm_levels,0.437389,0.347106,0.198264,0.008838,0.005390,0.205588
3,XGBoost,round2_plus_lm_levels,0.446040,0.347428,0.166240,0.008932,0.005406,0.188599
4,XGBoost,round2_baseline_price_context,0.448124,0.351321,0.158429,0.008939,0.005479,0.187285
5,XGBoost,round2_plus_text_change,0.453071,0.356953,0.139746,0.008955,0.005563,0.184462
6,ElasticNet,round2_baseline_price_context,0.474532,0.375767,0.056319,0.009651,0.005888,0.052614
7,ElasticNet,round2_plus_text_change,0.474814,0.375029,0.055197,0.009654,0.005888,0.052015
8,ElasticNet,round2_plus_lm_levels,0.475264,0.374334,0.053405,0.009694,0.005869,0.044259
9,LinearRegression,round2_plus_text_change,0.476406,0.376627,0.048850,0.009712,0.005928,0.040700


In [21]:
results_df.to_csv("data/11_round2_text_change/round2_model_results.csv", index=False)
print("Saved Round 2 results.")

Saved Round 2 results.


In [22]:
best_dataset = "round2_plus_text_change"

numeric_cols = [c for c in feature_sets[best_dataset] if c not in categorical_features]
categorical_cols = categorical_features.copy()

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ]
)

rf_best = Pipeline([
    ("prep", tree_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=400,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

y_train_best = train_df[target_col].copy()
rf_best.fit(train_df[feature_sets[best_dataset]], y_train_best)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['past_return_5d',
                                                   'past_return_10d',
                                                   'past_return_20d',
                                                   'abs_past_return_10d',
                                                   'past_realized_vol_5d',
                                                   'past_realized_vol_10d',
                                                   'past_realized_vol_20d',
                                                   'past_realized_vol_30d',
                                                   'past_realized_vol_60d',
                                                   'mean_abs_return_5d',
                                                   'mean_ab...
                                                   'delta_log_text_length_words',
                                                   'finbert_cosine_prev',
                                                   'finbert_l2_prev']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['quarter', 'filing_month',
                                                   'filing_year'])])),
                ('model',
                 RandomForestRegressor(max_depth=10, min_samples_leaf=5,
                                       n_estimators=400, n_jobs=-1,
                                       random_state=42))])

In [23]:
prep = rf_best.named_steps["prep"]
model = rf_best.named_steps["model"]

feature_names = prep.get_feature_names_out()
importances = model.feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_imp.head(25)

,feature,importance
10,num__mean_abs_return_10d,0.160632
6,num__past_realized_vol_20d,0.115402
8,num__past_realized_vol_60d,0.112911
7,num__past_realized_vol_30d,0.110157
9,num__mean_abs_return_5d,0.039318
2,num__past_return_20d,0.026713
0,num__past_return_5d,0.026626
35,cat__filing_month_2,0.025189
1,num__past_return_10d,0.021218
13,num__return_kurtosis_10d,0.020676


In [24]:
feat_imp.to_csv("data/11_round2_text_change/round2_feature_importance.csv", index=False)
print("Saved Round 2 feature importance.")

Saved Round 2 feature importance.


In [25]:
from google.colab import files

files.download("data/11_round2_text_change/round2_model_results.csv")
files.download("data/11_round2_text_change/round2_feature_importance.csv")
files.download("data/11_round2_text_change/round2_text_change_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>